# Student Data Cleaning / 学生数据清洗

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

data = pd.read_csv('/workspace/.uploads/23091186-5ec3-4346-9498-dc542bc22971_student_data_messy.csv')
data.head()


,Student_ID,Name,Age,Gender,Major,Attendance_pct,Homework,Presentation,Project,Enrollment_Date,Email,City,Height_cm,Weight_kg
0,1001,Ali Khan,20,Male,Computer Science,92.0,18.0,17.0,43,2026-02-15,ali@example.com,Wuhan,173,65.0
1,1002,Sara Li,19,F,computer science,88.0,19.0,18.0,45,15/02/2026,sara.li@example.com,Wuhan,165,52.0
2,1003,Zhang Wei,21,male,AI,95.0,20.0,19.0,48,2026/02/16,zhang.wei@example.com,Huangshi,178,70.0
3,1004,Fatima Noor,20,Female,Artificial Intelligence,NaN,16.0,17.0,40,"Feb 17, 2026",fatima.noor@example.com,Huangshi,160,50.0
4,1005,Chen Yu,twenty,M,CS,76.0,15.0,15.0,38,2026-02-18,chen.yu@example.com,wuhan,181,75.0


## 1. 列名规范化 / Clean column names

In [2]:
data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_', regex=False)
data.columns.tolist()


['student_id',
 'name',
 'age',
 'gender',
 'major',
 'attendance_pct',
 'homework',
 'presentation',
 'project',
 'enrollment_date',
 'email',
 'city',
 'height_cm',
 'weight_kg']

## 2. 缺失值符号统一 / Replace missing markers

In [3]:
data = data.replace(['', 'NA', 'N/A', '?', 'missing'], np.nan)
data.isna().sum()


student_id         0
name               0
age                0
gender             1
major              0
attendance_pct     1
homework           1
presentation       1
project            0
enrollment_date    0
email              0
city               0
height_cm          0
weight_kg          1
dtype: int64

## 3. 数值转换 / Convert numeric columns

In [4]:
numeric_cols = ['age','attendance_pct','homework','presentation','project','height_cm','weight_kg']
for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')
data[numeric_cols].dtypes


age               float64
attendance_pct    float64
homework          float64
presentation      float64
project             int64
height_cm           int64
weight_kg         float64
dtype: object

## 4. 范围校验 / Validate ranges

In [5]:
data.loc[~data['age'].between(16,80), 'age'] = np.nan
data.loc[~data['attendance_pct'].between(0,100), 'attendance_pct'] = np.nan
data.loc[~data['height_cm'].between(130,220), 'height_cm'] = np.nan
data.loc[~data['weight_kg'].between(35,200), 'weight_kg'] = np.nan


## 5. 日期解析 / Parse dates

In [6]:
data['enrollment_date'] = pd.to_datetime(data['enrollment_date'], errors='coerce')
data['enrollment_date'].head()


0   2026-02-15
1          NaT
2          NaT
3          NaT
4   2026-02-18
Name: enrollment_date, dtype: datetime64[us]

## 6. 处理缺失值 / Fill missing values

In [7]:
data['age'].fillna(data['age'].median(), inplace=True)
data['height_cm'].fillna(data['height_cm'].median(), inplace=True)
data['weight_kg'].fillna(data['weight_kg'].median(), inplace=True)
data['gender'].fillna(data['gender'].mode()[0], inplace=True)
data['major'].fillna(data['major'].mode()[0], inplace=True)
data['city'].fillna(data['city'].mode()[0], inplace=True)


/tmp/ipykernel_2711/982022707.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  data['age'].fillna(data['age'].median(), inplace=True)
/tmp/ipykernel_2711/982022707.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never work

0        Wuhan
1        Wuhan
2     Huangshi
3     Huangshi
4        wuhan
5        Wuhan
6     Huangshi
7     Huangshi
8        Wuhan
9     Huangshi
10       Wuhan
11       Wuhan
12    Huangshi
13    Huangshi
14       Wuhan
15    Huangshi
16       Wuhan
17       Wuhan
18    Huangshi
19    Huangshi
20       Wuhan
21       Wuhan
22    Huangshi
23    Huangshi
24       Wuhan
25       Wuhan
26    Huangshi
27    Huangshi
28       Wuhan
29       Wuhan
30       Wuhan
Name: city, dtype: str

## 7. 删除重复 / Remove duplicates

In [8]:
data = data.drop_duplicates()
data.shape


(30, 14)

## 8. 统一格式 / Standardize categories

In [9]:
data['gender'] = data['gender'].str.lower().map({'m':'Male','male':'Male','f':'Female','female':'Female'})
data['major'] = data['major'].str.lower().str.replace('.','').map({'cs':'Computer Science','computer science':'Computer Science','ai':'Artificial Intelligence','artificial intelligence':'Artificial Intelligence','data science':'Data Science'})
data['city'] = data['city'].str.title()


## 9. 邮箱清理 / Clean email

In [10]:
data['email'] = data['email'].str.strip().str.lower()
data['email_valid'] = data['email'].str.contains('@', na=False)


## 10. 编码 / Encoding

In [11]:
data = pd.get_dummies(data, columns=['city'], prefix='city')
le = LabelEncoder()
data['gender_encoded'] = le.fit_transform(data['gender'])
data['major_encoded'] = le.fit_transform(data['major'])


## 11. 数值缩放 / Scale numeric

In [12]:
scaler = StandardScaler()
data['age_scaled'] = scaler.fit_transform(data[['age']])


## 12. 保存 / Save

In [13]:
data.to_csv('/workspace/student_data_cleaned.csv', index=False)
print("处理后的数据：")
data.head()


处理后的数据：


,student_id,name,age,gender,major,attendance_pct,homework,presentation,project,enrollment_date,email,height_cm,weight_kg,email_valid,city_Huangshi,city_Wuhan,gender_encoded,major_encoded,age_scaled
0,1001,Ali Khan,20.0,Male,Computer Science,92.0,18.0,17.0,43,2026-02-15,ali@example.com,173.0,65.0,True,False,True,1,1,-0.427527
1,1002,Sara Li,19.0,Female,Computer Science,88.0,19.0,18.0,45,NaT,sara.li@example.com,165.0,52.0,True,False,True,0,1,-1.476912
2,1003,Zhang Wei,21.0,Male,Artificial Intelligence,95.0,20.0,19.0,48,NaT,zhang.wei@example.com,178.0,70.0,True,True,False,1,0,0.621858
3,1004,Fatima Noor,20.0,Female,Artificial Intelligence,NaN,16.0,17.0,40,NaT,fatima.noor@example.com,160.0,50.0,True,True,False,0,0,-0.427527
4,1005,Chen Yu,NaN,Male,Computer Science,76.0,15.0,15.0,38,2026-02-18,chen.yu@example.com,181.0,75.0,True,False,True,1,1,NaN
